# Human approval steps before risky actions

The office assistant of Linden Court, a 12-flat building, gets two jobs from the property manager: refund the
deposit of the tenant who moved out of flat 3B, minus damage, and email every tenant about a water shut-off.
In this notebook you build the lesson's LangGraph graph, pause it with `interrupt()` before the refund and the
email, and resume it as the manager: approve the refund, edit the email, and try a wrong thread id.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.
No model is called: the tool calls the model made are replayed from the lesson's recording
(google/gemini-2.5-flash, recorded 2026-11-13). The graph, the pause, the saving and the resuming are real LangGraph.

## 1. Install LangGraph

Colab does not have LangGraph. The next cell installs the version the lesson ran (1.2.21).

**Look for:** no error. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q langgraph==1.2.21

Note: you may need to restart the kernel to use updated packages.


## 2. The office records and the tools

These are the lesson's fake tools. Three only read records: `list_tenants`, `get_lease`, `get_inspection`.
Two change the world: `issue_refund` pays money and `send_email` sends mail. Here they only return what a real
system would report; nothing is paid or sent.

**Look for:** 12 tenants, a deposit of 1200, and damage of 150 + 40 = 190, so the right refund is 1010.

In [2]:
import json

MANAGER_MESSAGE = (
    "Two jobs for today. 1) The tenant of flat 3B moved out on 31 October and the move-out inspection is done: "
    "refund their deposit, minus any damage the inspection found. 2) Email every tenant that the water will be "
    "off on Thursday 20 November from 9:00 to 12:00 for a pipe repair."
)

TENANTS = [
    {"flat": f, "name": n, "email": e}
    for f, n, e in [
        ("1A", "Ana Petrov", "ana.p@example.com"), ("1B", "Tom Okafor", "tom.o@example.com"),
        ("2A", "Lea Brandt", "lea.b@example.com"), ("2B", "Ravi Menon", "ravi.m@example.com"),
        ("3A", "Sofia Ruiz", "sofia.r@example.com"), ("3B", "Jonas Berg", "jonas.b@example.com"),
        ("4A", "Mei Tanaka", "mei.t@example.com"), ("4B", "Omar Haddad", "omar.h@example.com"),
        ("5A", "Clara Weiss", "clara.w@example.com"), ("5B", "Ben Adler", "ben.a@example.com"),
        ("6A", "Nina Kovac", "nina.k@example.com"), ("6B", "Paul Grant", "paul.g@example.com"),
    ]
]
LEASES = {"3B": {"flat": "3B", "tenant": "Jonas Berg", "deposit": 1200, "currency": "EUR",
                 "moved_out": "2026-10-31", "refund_account": "IBAN ending 4471"}}
INSPECTIONS = {"3B": {"flat": "3B", "date": "2026-11-03", "damage": [
    {"item": "hole in hallway wall, filled and painted", "cost": 150},
    {"item": "one key not returned, lock re-keyed", "cost": 40}]}}


def run_tool(name, args):
    """Return the tool result as a JSON string, the way a real backend would."""
    if name == "list_tenants":
        return json.dumps(TENANTS)
    if name == "get_lease":
        f = args.get("flat", "")
        return json.dumps(LEASES.get(f, {"error": f"no lease for flat {f}"}))
    if name == "get_inspection":
        f = args.get("flat", "")
        return json.dumps(INSPECTIONS.get(f, {"error": f"no inspection for flat {f}"}))
    if name == "issue_refund":
        return json.dumps({"status": "paid", "flat": args.get("flat"), "amount": args.get("amount"), "payment_id": "PAY-30817"})
    if name == "send_email":
        return json.dumps({"status": "sent", "recipients": len(args.get("to", []))})
    return json.dumps({"error": f"unknown tool {name}"})


damage = sum(d["cost"] for d in INSPECTIONS["3B"]["damage"])
print("tenants:", len(TENANTS))
print("deposit:", LEASES["3B"]["deposit"], "| damage:", damage, "| right refund:", LEASES["3B"]["deposit"] - damage)

tenants: 12
deposit: 1200 | damage: 190 | right refund: 1010


## 3. The model's tool calls, from the recording

The model asked for five tool calls, one per turn. The cell holds them exactly as the model wrote them, then
turns each argument string into a Python dict.

**Look for:** five calls in this order: `get_lease`, `get_inspection`, `issue_refund` with amount 1010,
`list_tenants`, `send_email`. The model's last message says it did both jobs.

In [3]:
# Copied exactly from the recording flash-1..5: (tool name, argument string the model wrote)
RECORDED = [
    ('get_lease',
     '{"flat":"3B"}'),
    ('get_inspection',
     '{"flat":"3B"}'),
    ('issue_refund',
     '{"amount":1010,"reason":"Deposit refund, less EUR 150 for a hole in the hallway wall and EUR 40 for one key not returned.","flat":"3B"}'),
    ('list_tenants',
     '{}'),
    ('send_email',
     '{"body":"Dear tenants,\\n\\nPlease be advised that the water will be off on Thursday, November 20th, from 9:00 to 12:00 for a pipe repair.\\n\\nThank you for your understanding.\\n\\nSincerely,\\nLinden Court Management","subject":"Water Shutoff Notification","to":["ana.p@example.com","tom.o@example.com","lea.b@example.com","ravi.m@example.com","sofia.r@example.com","jonas.b@example.com","mei.t@example.com","omar.h@example.com","clara.w@example.com","ben.a@example.com","nina.k@example.com","paul.g@example.com"]}'),
]
FINAL_MESSAGE = 'The deposit for flat 3B has been refunded, with EUR 190 deducted for damages. An email has been sent to all tenants regarding a water shutoff on November 20th.'

RECORDED_CALLS = [{"name": n, "args": json.loads(a)} for n, a in RECORDED]
print("recorded tool calls:", len(RECORDED_CALLS))
for i, c in enumerate(RECORDED_CALLS, 1):
    print(f"  {i}. {c['name']}  {json.dumps(c['args'])[:80]}")
print("the model's last message:", FINAL_MESSAGE)

recorded tool calls: 5
  1. get_lease  {"flat": "3B"}
  2. get_inspection  {"flat": "3B"}
  3. issue_refund  {"amount": 1010, "reason": "Deposit refund, less EUR 150 for a hole in the hallw
  4. list_tenants  {}
  5. send_email  {"body": "Dear tenants,\n\nPlease be advised that the water will be off on Thurs
the model's last message: The deposit for flat 3B has been refunded, with EUR 190 deducted for damages. An email has been sent to all tenants regarding a water shutoff on November 20th.


## 4. The run with nobody looking

This is what happened in the lesson's first screen: each call runs the moment the model asks.

**Look for:** the refund is paid (`PAY-30817`) and the email goes to **12** addresses. One of them,
`jonas.b@example.com`, is Jonas Berg, the tenant who moved out of 3B. Nobody checked.

In [4]:
for c in RECORDED_CALLS:
    print(f"{c['name']}: {run_tool(c['name'], c['args'])[:110]}")

to = RECORDED_CALLS[4]["args"]["to"]
print()
print("addresses in send_email:", len(to))
print("jonas.b@example.com in the list:", "jonas.b@example.com" in to)

get_lease: {"flat": "3B", "tenant": "Jonas Berg", "deposit": 1200, "currency": "EUR", "moved_out": "2026-10-31", "refund_
get_inspection: {"flat": "3B", "date": "2026-11-03", "damage": [{"item": "hole in hallway wall, filled and painted", "cost": 1
issue_refund: {"status": "paid", "flat": "3B", "amount": 1010, "payment_id": "PAY-30817"}
list_tenants: [{"flat": "1A", "name": "Ana Petrov", "email": "ana.p@example.com"}, {"flat": "1B", "name": "Tom Okafor", "ema
send_email: {"status": "sent", "recipients": 12}

addresses in send_email: 12
jonas.b@example.com in the list: True


## 5. The graph, with a pause before risky tools

Two nodes. `model` stands in for the model: it hands back the next recorded tool call. `run_tools` runs the
calls. For a tool in `NEEDS_SIGN_OFF` it first calls `interrupt(...)` with the tool and its arguments: that is the
card the manager sees. The checkpointer `InMemorySaver` saves the paused run under a thread id.

This is the lesson's code, line for line. **Look for:** the graph's nodes printed at the end.

In [5]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

NEEDS_SIGN_OFF = {"issue_refund", "send_email"}


class State(TypedDict):
    turn: int
    pending: list
    log: Annotated[list, operator.add]


def model(state: State):
    """Stand-in for the model: hands back the next recorded tool call."""
    t = state["turn"]
    return {"turn": t + 1, "pending": [RECORDED_CALLS[t]] if t < len(RECORDED_CALLS) else []}


def run_tools(state):
    done = []
    for call in state["pending"]:
        args = call["args"]
        if call["name"] in NEEDS_SIGN_OFF:
            decision = interrupt({"tool": call["name"], "args": args})
            if decision["type"] == "reject":
                done.append(f"{call['name']}: rejected by the manager")
                continue
            args = decision.get("args", args)
        done.append(f"{call['name']}: {run_tool(call['name'], args)}")
    return {"log": done}


def more(state: State):
    return "run_tools" if state["pending"] else END


builder = StateGraph(State)
builder.add_node("model", model)
builder.add_node("run_tools", run_tools)
builder.add_edge(START, "model")
builder.add_conditional_edges("model", more, ["run_tools", END])
builder.add_edge("run_tools", "model")
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "flat-3B-and-water-notice"}}
print("nodes:", [n for n in graph.get_graph().nodes])

nodes: ['__start__', 'model', 'run_tools', '__end__']


## 6. Start the run: it pauses before the refund

`show` prints the paused call (from `__interrupt__`) and the log. The two reads run on their own.

**Look for:** `PAUSED` with `issue_refund` and amount 1010, and a log with only `get_lease` and
`get_inspection`: no payment yet. Then `next` is `('run_tools',)`, the node that runs on resume.

In [6]:
def show(label, result):
    print(f"=== {label}")
    for it in result.get("__interrupt__", []):
        v = it.value
        if "to" in v["args"]:
            print("PAUSED:", v["tool"], "| to:", len(v["args"]["to"]), "addresses")
        else:
            print("PAUSED:", v["tool"], "| args:", json.dumps(v["args"]))
    print("log:")
    for line in result.get("log", []):
        print("  ", line[:100])


r = graph.invoke({"turn": 0, "pending": [], "log": []}, config)
show("invoke #1 (start)", r)
print("state.next:", graph.get_state(config).next)

=== invoke #1 (start)
PAUSED: issue_refund | args: {"amount": 1010, "reason": "Deposit refund, less EUR 150 for a hole in the hallway wall and EUR 40 for one key not returned.", "flat": "3B"}
log:
   get_lease: {"flat": "3B", "tenant": "Jonas Berg", "deposit": 1200, "currency": "EUR", "moved_out": "
   get_inspection: {"flat": "3B", "date": "2026-11-03", "damage": [{"item": "hole in hallway wall, fill
state.next: ('run_tools',)


## 7. Resume on the wrong thread id

By mistake the code resumes with `"some-other-thread"`. Nothing was saved under that id, so the graph starts from
an empty state, and the `model` node fails when it reads `state["turn"]`. The cell catches the error so the
notebook keeps going.

**Look for:** `KeyError` with `'turn'`. The paused refund on the real thread is untouched.

In [7]:
try:
    graph.invoke(Command(resume={"type": "approve"}), {"configurable": {"thread_id": "some-other-thread"}})
except Exception as e:
    print("EXCEPTION:", type(e).__name__, e)

print("real thread still waiting at:", graph.get_state(config).next)

EXCEPTION: KeyError 'turn'
real thread still waiting at: ('run_tools',)


## 8. The manager approves the refund

1200 − 150 − 40 = 1010, the amount on the card. She resumes the same thread with `{"type": "approve"}`.

**Look for:** the log now has the `issue_refund` line with `PAY-30817`, then `list_tenants`, and the run pauses
again at `send_email` with 12 addresses.

In [8]:
r = graph.invoke(Command(resume={"type": "approve"}), config)
show("invoke #2 (approve refund)", r)

=== invoke #2 (approve refund)
PAUSED: send_email | to: 12 addresses
log:
   get_lease: {"flat": "3B", "tenant": "Jonas Berg", "deposit": 1200, "currency": "EUR", "moved_out": "
   get_inspection: {"flat": "3B", "date": "2026-11-03", "damage": [{"item": "hole in hallway wall, fill
   issue_refund: {"status": "paid", "flat": "3B", "amount": 1010, "payment_id": "PAY-30817"}
   list_tenants: [{"flat": "1A", "name": "Ana Petrov", "email": "ana.p@example.com"}, {"flat": "1B", "n


## 9. The manager edits the email

The notice is right, one address is wrong. She takes the paused call's arguments, removes Jonas Berg's address,
and resumes with `{"type": "edit", "args": ...}`.

**Look for:** the edited list has 11 addresses, `send_email` reports `"recipients": 11`, and `next` is `()`:
the run is finished.

In [9]:
paused = r["__interrupt__"][0].value
new_to = [a for a in paused["args"]["to"] if a != "jonas.b@example.com"]
edited = dict(paused["args"], to=new_to)
print("edited to-list length:", len(new_to), "| removed:", sorted(set(paused["args"]["to"]) - set(new_to)))

r = graph.invoke(Command(resume={"type": "edit", "args": edited}), config)
show("invoke #3 (edit email)", r)
print("state.next:", graph.get_state(config).next)

edited to-list length: 11 | removed: ['jonas.b@example.com']
=== invoke #3 (edit email)
log:
   get_lease: {"flat": "3B", "tenant": "Jonas Berg", "deposit": 1200, "currency": "EUR", "moved_out": "
   get_inspection: {"flat": "3B", "date": "2026-11-03", "damage": [{"item": "hole in hallway wall, fill
   issue_refund: {"status": "paid", "flat": "3B", "amount": 1010, "payment_id": "PAY-30817"}
   list_tenants: [{"flat": "1A", "name": "Ana Petrov", "email": "ana.p@example.com"}, {"flat": "1B", "n
   send_email: {"status": "sent", "recipients": 11}
state.next: ()


## 10. The node runs again from the top

On resume, LangGraph does not continue after `interrupt()`. It runs `run_tools` again from its first line, and this
time `interrupt()` returns the manager's answer. The cell records which lines of `run_tools` run for the
`issue_refund` call, on a new thread (line 0 is the `def` line, as on the lesson's screen).

**Look for:** in invoke #1 the lines stop at 5 (the `interrupt` line). In invoke #2 lines 1 to 5 run again, then
6, 9, 10. Line 10, `run_tool`, runs once: the refund is paid once, after the yes.

In [10]:
import sys, threading

FIRST = run_tools.__code__.co_firstlineno
LINES = []


def _local(frame, event, arg):
    if event == "line":
        call = frame.f_locals.get("call")
        if call is None or call["name"] == "issue_refund":
            LINES.append(frame.f_lineno - FIRST)
    return _local


def _global(frame, event, arg):
    return _local if frame.f_code is run_tools.__code__ else None


def lines_of_refund(thunk):
    """Run thunk with the tracer on; keep only the run_tools invocation that handled issue_refund."""
    LINES.clear()
    sys.settrace(_global); threading.settrace(_global)
    try:
        thunk()
    finally:
        sys.settrace(None); threading.settrace(None)
    starts = [i for i, l in enumerate(LINES) if l == 1] + [len(LINES)]
    runs = [LINES[a:b] for a, b in zip(starts, starts[1:])]
    return [run for run in runs if 3 in run and 4 in run and (5 in run)][-1]


cfg = {"configurable": {"thread_id": "trace-demo"}}
first = lines_of_refund(lambda: graph.invoke({"turn": 0, "pending": [], "log": []}, cfg))
second = lines_of_refund(lambda: graph.invoke(Command(resume={"type": "approve"}), cfg))
print("invoke #1, lines run:", first)
print("invoke #2, lines run:", second)
print("lines run in both:", sorted(set(first) & set(second)))
print("run_tool line 10 ran in invoke #1:", 10 in first, "| in invoke #2:", second.count(10), "time(s)")

invoke #1, lines run: [1, 2, 3, 4, 5]
invoke #2, lines run: [1, 2, 3, 4, 5, 6, 9, 10, 2, 11]
lines run in both: [1, 2, 3, 4, 5]
run_tool line 10 ran in invoke #1: False | in invoke #2: 1 time(s)


## Now change it

Each cell below has one line marked `# <- change this`. Run it once as it is, then edit that line and run it again.

### Experiment 1: what needs a yes

The cell runs the whole job on a new thread and answers every pause with "approve".

**What should change:** as it is, the run pauses twice (`issue_refund`, `send_email`). Remove `"send_email"` from
the set and it pauses only once, and the email goes to all 12 addresses with nobody looking, as in step 4.
The stop lives in the code, so the set of tool names decides everything.

In [11]:
NEEDS_SIGN_OFF = {"issue_refund", "send_email"}  # <- change this (try {"issue_refund"})


def run_job(thread_id, answer):
    cfg = {"configurable": {"thread_id": thread_id}}
    r = graph.invoke({"turn": 0, "pending": [], "log": []}, cfg)
    pauses = []
    while "__interrupt__" in r:
        pauses.append(r["__interrupt__"][0].value["tool"])
        r = graph.invoke(Command(resume=answer), cfg)
    return pauses, r["log"]


pauses, log = run_job("experiment-1-" + "-".join(sorted(NEEDS_SIGN_OFF)), {"type": "approve"})
print("paused before:", pauses)
print("last log line:", log[-1])
NEEDS_SIGN_OFF = {"issue_refund", "send_email"}  # put the lesson's set back for the next cells

paused before: ['issue_refund', 'send_email']
last log line: send_email: {"status": "sent", "recipients": 12}


### Experiment 2: the manager's answer

The cell starts a new thread and answers the refund card with `decision`.

**What should change:** with `approve` the log shows the payment with amount 1010. With `{"type": "reject"}` the
tool never runs and the log says `issue_refund: rejected by the manager`. With
`{"type": "edit", "args": {"flat": "3B", "amount": 1200, "reason": "full deposit"}}` the tool pays 1200: an edit
replaces the model's arguments completely, so a wrong edit is paid too.

In [12]:
decision = {"type": "approve"}  # <- change this (try {"type": "reject"})

cfg = {"configurable": {"thread_id": "experiment-2-" + json.dumps(decision)}}
graph.invoke({"turn": 0, "pending": [], "log": []}, cfg)
r = graph.invoke(Command(resume=decision), cfg)
print("refund line in the log:", [l for l in r["log"] if l.startswith("issue_refund")][0])
print("now paused before:", r["__interrupt__"][0].value["tool"])

refund line in the log: issue_refund: {"status": "paid", "flat": "3B", "amount": 1010, "payment_id": "PAY-30817"}
now paused before: send_email


### Experiment 3: a side effect before the pause

This cell builds a second version of the node that can pay **before** asking, and counts how many payments the
tool makes for one approved refund.

**What should change:** with `False` (pay after the yes, as in the lesson) there is 1 payment. With `True` there
are 2: the first invoke pays and then pauses, and the resume runs the node from the top and pays again. Anything
before `interrupt()` runs twice.

In [13]:
PAY_BEFORE_ASKING = False  # <- change this (try True)

payments = []


def run_tools_v2(state):
    done = []
    for call in state["pending"]:
        if call["name"] == "issue_refund" and PAY_BEFORE_ASKING:
            payments.append(call["args"]["amount"])                 # pays before the yes
        if call["name"] in NEEDS_SIGN_OFF:
            decision = interrupt({"tool": call["name"], "args": call["args"]})
        if call["name"] == "issue_refund" and not PAY_BEFORE_ASKING:
            payments.append(call["args"]["amount"])                 # pays after the yes
        done.append(f"{call['name']}: {run_tool(call['name'], call['args'])}")
    return {"log": done}


b2 = StateGraph(State)
b2.add_node("model", model)
b2.add_node("run_tools", run_tools_v2)
b2.add_edge(START, "model")
b2.add_conditional_edges("model", more, ["run_tools", END])
b2.add_edge("run_tools", "model")
graph2 = b2.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "experiment-3"}}
graph2.invoke({"turn": 0, "pending": [], "log": []}, cfg)
print("payments after invoke #1 (paused):", payments)
graph2.invoke(Command(resume={"type": "approve"}), cfg)
print("payments after the approve:", payments)
print("number of payments:", len(payments))

payments after invoke #1 (paused): []
payments after the approve: [1010]
number of payments: 1


## What you should see

- Step 6: the first `invoke` returns a result with `__interrupt__`; its value names `issue_refund` with amount 1010, and the log has no payment yet.
- Step 6: while paused, `graph.get_state(config).next` prints `('run_tools',)`.
- Step 8: resuming with `{"type": "approve"}` adds the `issue_refund` line with `PAY-30817`, then pauses again at `send_email` with 12 addresses.
- Step 9: resuming with the edited list (without jonas.b@example.com) makes `send_email` report 11 recipients, and `next` is then `()`.
- Step 7: resuming on another thread id fails with a `KeyError` for `'turn'`.
- Step 10: for the refund, lines `[1, 2, 3, 4, 5]` of `run_tools` run in both invokes, and line 10 (`run_tool`) runs once.